In [ ]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (6) (1).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [ ]:
df["Оценка"].value_counts()

,count
Оценка,
4,208
3,206
1,110
2,105
5,94
7,26
6,21
8,21
9,18


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
import pickle

In [ ]:
y_audience = df['ЦА'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text, y_audience, test_size=0.2, random_state=42
)
tfidf_audience = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_audience = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_audience = tfidf_audience.fit_transform(X_train_text_audience)
X_train_svd_audience = svd_audience.fit_transform(X_train_tfidf_audience)
X_test_tfidf_audience = tfidf_audience.transform(X_test_text_audience)
X_test_svd_audience = svd_audience.transform(X_test_tfidf_audience)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
model_audience = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)

In [ ]:
model_audience.fit(X_train_svd_audience , y_train_audience)

0:	learn: 2.2454054	total: 130ms	remaining: 38.8s
50:	learn: 1.1172123	total: 1.87s	remaining: 9.11s
100:	learn: 0.7256858	total: 3.58s	remaining: 7.05s
150:	learn: 0.5856018	total: 5.93s	remaining: 5.85s
200:	learn: 0.5059283	total: 7.59s	remaining: 3.74s
250:	learn: 0.4499342	total: 9.25s	remaining: 1.8s
299:	learn: 0.4091207	total: 10.9s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=3, early_stopping_rounds=50, eval_metric='MultiClass', iterations=300, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=50)

In [ ]:
y_pred_audience = model_audience.predict(X_test_svd_audience)
print(classification_report(y_test_audience, y_pred_audience))
print("MAE=",mean_absolute_error(y_test_audience, y_pred_audience))

              precision    recall  f1-score   support

           1       0.50      0.54      0.52        13
           2       0.50      0.27      0.35        22
           3       0.43      0.44      0.43        34
           4       0.53      0.40      0.46        52
           5       0.43      0.69      0.53        32
           6       0.00      0.00      0.00         2
           7       0.00      0.00      0.00         3
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         4

    accuracy                           0.44       163
   macro avg       0.26      0.26      0.25       163
weighted avg       0.45      0.44      0.43       163

MAE= 0.8159509202453987


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
y_sol= df['Проработка решения'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text, y_sol, test_size=0.2, random_state=42
)
tfidf_sol = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_sol= TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_sol = tfidf_sol.fit_transform(X_train_text_sol)
X_train_svd_sol = svd_sol.fit_transform(X_train_tfidf_sol)
X_test_tfidf_sol = tfidf_sol.transform(X_test_text_sol)
X_test_svd_sol = svd_sol.transform(X_test_tfidf_sol)

In [ ]:
model_sol = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_sol.fit(X_train_svd_sol , y_train_sol)
y_pred_sol = model_sol.predict(X_test_svd_sol)
#y_pred_sol
print(classification_report(y_test_sol, y_pred_sol))
print("MAE=",mean_absolute_error(y_test_sol, y_pred_sol))

0:	learn: 2.2303640	total: 49.9ms	remaining: 14.9s
50:	learn: 1.1967234	total: 2.31s	remaining: 11.3s
100:	learn: 0.7603683	total: 4.19s	remaining: 8.25s
150:	learn: 0.5988641	total: 5.86s	remaining: 5.78s
200:	learn: 0.5043770	total: 7.51s	remaining: 3.7s
250:	learn: 0.4519465	total: 9.16s	remaining: 1.79s
299:	learn: 0.4057601	total: 11.2s	remaining: 0us
              precision    recall  f1-score   support

           1       0.60      0.60      0.60        15
           2       0.35      0.29      0.32        24
           3       0.55      0.35      0.43        31
           4       0.43      0.52      0.47        46
           5       0.45      0.49      0.47        37
           6       0.00      0.00      0.00         2
           7       0.38      0.75      0.50         4
           8       0.00      0.00      0.00         2
           9       0.00      0.00      0.00         2

    accuracy                           0.44       163
   macro avg       0.31      0.33      0.31  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
y_finance= df['Финансовая модель и метрики'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_finance, X_test_text_finance,y_train_finance, y_test_finance = train_test_split(
    X_text, y_finance, test_size=0.2, random_state=42
)
tfidf_finance = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_finance = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_finance = tfidf_finance.fit_transform(X_train_text_finance)
X_train_svd_finance = svd_finance.fit_transform(X_train_tfidf_finance)
X_test_tfidf_finance = tfidf_finance.transform(X_test_text_finance)
X_test_svd_finance = svd_finance.transform(X_test_tfidf_finance)

In [ ]:
model_finance = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_finance.fit(X_train_svd_finance , y_train_finance)
y_pred_finance = model_finance.predict(X_test_svd_finance)
print(classification_report(y_test_finance, y_pred_finance))
print("MAE=",mean_absolute_error(y_test_finance, y_pred_finance))

0:	learn: 2.2423617	total: 49.7ms	remaining: 14.9s
50:	learn: 1.1455268	total: 1.76s	remaining: 8.57s
100:	learn: 0.7178146	total: 3.43s	remaining: 6.76s
150:	learn: 0.5749159	total: 5.41s	remaining: 5.34s
200:	learn: 0.4853959	total: 7.8s	remaining: 3.84s
250:	learn: 0.4339613	total: 9.53s	remaining: 1.86s
299:	learn: 0.3970890	total: 11.2s	remaining: 0us
              precision    recall  f1-score   support

           1       0.47      0.75      0.58        12
           2       0.30      0.21      0.24        29
           3       0.50      0.38      0.43        37
           4       0.50      0.44      0.47        50
           5       0.42      0.73      0.54        26
           6       0.20      0.33      0.25         3
           7       0.00      0.00      0.00         2
           8       0.00      0.00      0.00         2
           9       0.00      0.00      0.00         2

    accuracy                           0.44       163
   macro avg       0.27      0.32      0.28  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
y_risks= df['Анализ рисков'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text, y_risks, test_size=0.2, random_state=42
)
tfidf_risks = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_risks = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_risks = tfidf_risks.fit_transform(X_train_text_risks)
X_train_svd_risks = svd_risks.fit_transform(X_train_tfidf_risks)
X_test_tfidf_risks = tfidf_risks.transform(X_test_text_risks)
X_test_svd_risks = svd_risks.transform(X_test_tfidf_risks)

In [ ]:
model_risks = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_risks.fit(X_train_svd_risks , y_train_risks)
y_pred_risks = model_risks.predict(X_test_svd_risks)
print(classification_report(y_test_risks, y_pred_risks))
print("MAE=",mean_absolute_error(y_test_risks, y_pred_risks))

0:	learn: 2.2481032	total: 88.2ms	remaining: 26.4s
50:	learn: 1.1246167	total: 2.13s	remaining: 10.4s
100:	learn: 0.7047727	total: 3.87s	remaining: 7.62s
150:	learn: 0.5509476	total: 5.59s	remaining: 5.52s
200:	learn: 0.4740904	total: 7.25s	remaining: 3.57s
250:	learn: 0.4195425	total: 8.93s	remaining: 1.74s
299:	learn: 0.3747273	total: 10.6s	remaining: 0us
              precision    recall  f1-score   support

           1       0.56      0.75      0.64        20
           2       0.42      0.30      0.35        27
           3       0.62      0.53      0.57        47
           4       0.32      0.41      0.36        32
           5       0.30      0.29      0.29        28
           6       0.33      1.00      0.50         2
           7       0.00      0.00      0.00         3
           8       1.00      0.67      0.80         3
           9       0.00      0.00      0.00         1

    accuracy                           0.45       163
   macro avg       0.39      0.44      0.39 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
y_proves= df['Доказательства'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text, y_proves, test_size=0.2, random_state=42
)
tfidf_proves = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_proves = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_proves = tfidf_proves.fit_transform(X_train_text_proves)
X_train_svd_proves = svd_proves.fit_transform(X_train_tfidf_proves)
X_test_tfidf_proves = tfidf_proves.transform(X_test_text_proves)
X_test_svd_proves = svd_proves.transform(X_test_tfidf_proves)

In [ ]:
model_proves = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_proves.fit(X_train_svd_proves , y_train_proves)
y_pred_proves = model_proves.predict(X_test_svd_proves)
print(classification_report(y_test_proves, y_pred_proves))
print("MAE=",mean_absolute_error(y_test_proves, y_pred_proves))

0:	learn: 2.2592245	total: 50.4ms	remaining: 15.1s
50:	learn: 1.1473447	total: 1.75s	remaining: 8.56s
100:	learn: 0.7188690	total: 3.43s	remaining: 6.76s
150:	learn: 0.5551869	total: 5.1s	remaining: 5.03s
200:	learn: 0.4764960	total: 6.78s	remaining: 3.34s
250:	learn: 0.4222024	total: 8.43s	remaining: 1.65s
299:	learn: 0.3824317	total: 10.8s	remaining: 0us
              precision    recall  f1-score   support

           1       0.78      0.67      0.72        27
           2       0.45      0.45      0.45        22
           3       0.48      0.30      0.37        37
           4       0.49      0.47      0.48        40
           5       0.33      0.57      0.42        28
           6       0.00      0.00      0.00         2
           7       0.00      0.00      0.00         3
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         3

    accuracy                           0.45       163
   macro avg       0.28      0.27      0.27  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [ ]:
def get_prediction_audience(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_audience.transform([text])
  text_svd = svd_audience.transform(text_tfidf)
  score = model_audience.predict(text_svd)[0]
  return int(score)

In [ ]:
def get_prediction_solution(text):
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_sol.transform([text])
  text_svd = svd_sol.transform(text_tfidf)
  score = model_sol.predict(text_svd)[0]
  return int(score)

In [ ]:
def get_prediction_finance(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_finance.transform([text])
  text_svd = svd_finance.transform(text_tfidf)
  score = model_finance.predict(text_svd)[0]
  return int(score)

In [ ]:
def get_prediction_risks(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_risks.transform([text])
  text_svd = svd_risks.transform(text_tfidf)
  score = model_risks.predict(text_svd)[0]
  return int(score)

In [ ]:
def get_prediction_proves(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_proves.transform([text])
  text_svd = svd_proves.transform(text_tfidf)
  score = model_proves.predict(text_svd)[0]
  return int(score)

In [ ]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

/tmp/ipykernel_5396/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_5396/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_5396/131350308.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_5396/2249487333.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [ ]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.5461538461538461
f1micro_audience=  0.5461538461538461
f1macro_audience=  0.5505071343078847
              precision    recall  f1-score   support

           1       0.67      0.78      0.72        23
           2       0.65      0.50      0.57        26
           3       0.52      0.41      0.46        34
           4       0.43      0.55      0.48        22
           5       0.50      0.56      0.53        25

    accuracy                           0.55       130
   macro avg       0.55      0.56      0.55       130
weighted avg       0.55      0.55      0.54       130

MAE= 0.6307692307692307


In [ ]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.4230769230769231
f1micro_sol=  0.4230769230769231
f1macro_sol=  0.39430911783852957
              precision    recall  f1-score   support

           1       0.45      0.95      0.62        21
           2       0.58      0.34      0.43        32
           3       0.50      0.23      0.32        30
           4       0.37      0.56      0.44        25
           5       0.20      0.14      0.16        22

    accuracy                           0.42       130
   macro avg       0.42      0.45      0.39       130
weighted avg       0.44      0.42      0.39       130

MAE= 0.8153846153846154


In [ ]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.5461538461538461
f1micro_finance=  0.5461538461538461
f1macro_finance=  0.5286549707602338
              precision    recall  f1-score   support

           1       0.57      1.00      0.73        23
           2       0.50      0.36      0.42        33
           3       0.55      0.19      0.29        31
           4       0.51      0.83      0.63        24
           5       0.62      0.53      0.57        19

    accuracy                           0.55       130
   macro avg       0.55      0.58      0.53       130
weighted avg       0.54      0.55      0.50       130

MAE= 0.5692307692307692


In [ ]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.4230769230769231
f1micro_risks=  0.4230769230769231
f1macro_risks=  0.3843171207877091
              precision    recall  f1-score   support

           1       0.43      0.73      0.54        26
           2       0.52      0.38      0.44        32
           3       0.29      0.20      0.24        30
           4       0.41      0.59      0.48        27
           5       0.67      0.13      0.22        15

    accuracy                           0.42       130
   macro avg       0.46      0.41      0.38       130
weighted avg       0.44      0.42      0.40       130

MAE= 0.823076923076923


In [ ]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.4846153846153846
f1micro_proves=  0.4846153846153846
f1macro_proves=  0.46272525619168087
              precision    recall  f1-score   support

           1       0.38      0.83      0.52        18
           2       0.68      0.50      0.58        38
           3       0.31      0.16      0.21        25
           4       0.40      0.53      0.45        19
           5       0.62      0.50      0.56        30

    accuracy                           0.48       130
   macro avg       0.48      0.50      0.46       130
weighted avg       0.51      0.48      0.48       130

MAE 0.7230769230769231


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [ ]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [ ]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.5769230769230769


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
